In [ ]:
# Core Python utilities
import os                # For directory and file handling
import numpy as np       # For numerical operations and array manipulation
import shutil            # For file operations like copying/moving
import pandas as pd      # For structured data handling (CSV, tabular data)
import matplotlib.pyplot as plt  # For plotting graphs
import seaborn as sns    # For statistical data visualization
import itertools         # For efficient looping and combinations

# Deep learning framework
import tensorflow        # TensorFlow backend

# Image preprocessing
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# Helps with augmenting and loading image datasets

# Model architecture
from tensorflow.keras.models import Sequential, model_from_json
# Sequential: stack layers linearly
# model_from_json: reload saved model architecture

# Layers
from tensorflow.keras.layers import Dense          # Fully connected layer
from tensorflow.keras.layers import Conv2D         # Convolutional layer
from tensorflow.keras.layers import MaxPool2D, Dropout, MaxPooling2D
# MaxPool2D/MaxPooling2D: downsampling feature maps
# Dropout: regularization to prevent overfitting

from tensorflow.keras.layers import Flatten        # Flatten 2D features into 1D for Dense layers

# Training callbacks
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
# EarlyStopping: stop training when validation loss stops improving
# ModelCheckpoint: save best model weights during training

In [ ]:
pip install kaggle 
#This installs the official Kaggle API client into your Python environment

In [ ]:
from google.colab import files
files.upload()    #to upload files from your local machine into the Colab environment.

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sarothiadhikary","key":"9ea7b02a98becb31e34f81fccf9f149d"}'}

In [ ]:
!mkdir -p /root/.kaggle  #To create hidden directory inside root

In [ ]:
!mv kaggle.json /root/.kaggle/  #Kaggle API credentials file into hidden directory under root

In [ ]:
!chmod 600 /root/.kaggle/kaggle.json  #This sets the file permissions for Kaggle API credentials 

In [ ]:
!kaggle datasets download -d paultimothymooney/breast-histopathology-images   #download dataset

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/breast-histopathology-images
License(s): CC0-1.0
 99% 3.08G/3.10G [00:27<00:00, 180MB/s]
100% 3.10G/3.10G [00:27<00:00, 122MB/s]


In [ ]:
!unzip breast-histopathology-images.zip  #unzip folder

In [ ]:
import os   # Import the os module for interacting with the operating system (paths, directories)

# Define the path to the main dataset directory containing patient subfolders
cancer_rays_dir_str = "/content/IDC_regular_ps50_idx5"

# List all entries (patients) inside the dataset directory
cancer_rays_dir     = os.listdir(cancer_rays_dir_str)

# Define a new directory where all images will be collected together
all_rays_dir = "/content/all_rays_dir"

# Create the directory if it doesn’t already exist (avoid errors if rerun)
os.makedirs(all_rays_dir, exist_ok=True)

In [ ]:
print(cancer_rays_dir)
print(len(cancer_rays_dir))

In [11]:
os.makedirs(all_rays_dir, exist_ok=True)
all_rays_dir_lst = os.listdir(all_rays_dir)

In [ ]:
import os, shutil   # os for path operations, shutil for file copying

# Loop through each patient folder inside the cancer_rays_dir
for patient in cancer_rays_dir:

    # Construct paths for class "0" and class "1" subfolders for this patient
    path_0 = os.path.join(cancer_rays_dir_str, str(patient), "0")
    path_1 = os.path.join(cancer_rays_dir_str, str(patient), "1")

    # Skip if either path is not a directory (important to avoid errors in Colab)
    if not os.path.isdir(path_0) or not os.path.isdir(path_1):
        continue

    # Get list of all files inside the "0" and "1" folders
    file_list_0 = os.listdir(path_0)
    file_list_1 = os.listdir(path_1)

    # Copy each file from the "0" folder into the unified all_rays_dir
    for fname in file_list_0:
        src = os.path.join(path_0, fname)   # source file path
        dst = os.path.join(all_rays_dir, fname)  # destination path
        shutil.copyfile(src, dst)           # copy file

    # Copy each file from the "1" folder into the unified all_rays_dir
    for fname in file_list_1:
        src = os.path.join(path_1, fname)   # source file path
        dst = os.path.join(all_rays_dir, fname)  # destination path
        shutil.copyfile(src, dst)           # copy file


In [13]:
all_rays_dir_lst = os.listdir(all_rays_dir)
len(all_rays_dir_lst)

277524

In [14]:
data = pd.DataFrame(all_rays_dir_lst, columns=['image_id'])
data.head()

,image_id
0,10257_idx5_x1751_y551_class0.png
1,10268_idx5_x501_y601_class0.png
2,12906_idx5_x1851_y2401_class1.png
3,14156_idx5_x2351_y1651_class0.png
4,14079_idx5_x3101_y1201_class0.png


In [15]:
def extract_target(x):
    try:
        a = x.split('_')
        b = a[4]
        target = b[5]
        return target
    except IndexError:
        # If filename format is unexpected, assign a default label
        return -1  # or 'unknown'

data['target'] = data['image_id'].apply(extract_target)

data.head(10)

,image_id,target
0,10257_idx5_x1751_y551_class0.png,0
1,10268_idx5_x501_y601_class0.png,0
2,12906_idx5_x1851_y2401_class1.png,1
3,14156_idx5_x2351_y1651_class0.png,0
4,14079_idx5_x3101_y1201_class0.png,0
5,10276_idx5_x1251_y1451_class1.png,1
6,10268_idx5_x1551_y451_class0.png,0
7,14155_idx5_x1701_y951_class1.png,1
8,16570_idx5_x1601_y601_class0.png,0
9,10272_idx5_x3351_y951_class0.png,0


In [16]:
def extract_patient_id(x):
    try:
        a = x.split('_')
        patient_id = a[0]
        return patient_id
    except IndexError:
        return 'unknown'  # fallback for unexpected filenames

data['patient_id'] = data['image_id'].apply(extract_patient_id)
data.head()

,image_id,target,patient_id
0,10257_idx5_x1751_y551_class0.png,0,10257
1,10268_idx5_x501_y601_class0.png,0,10268
2,12906_idx5_x1851_y2401_class1.png,1,12906
3,14156_idx5_x2351_y1651_class0.png,0,14156
4,14079_idx5_x3101_y1201_class0.png,0,14079


In [17]:
data['target'].value_counts()

,count
target,
0,198738
1,78786


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

# Ensure target column is integer
data['target'] = data['target'].astype(int)

# Prepare figure
fig, ax = plt.subplots(5, 10, figsize=(20, 10))

# Random selection of positive and negative samples
pos_selection = np.random.choice(data[data.target == 1].index, size=50, replace=False)
neg_selection = np.random.choice(data[data.target == 0].index, size=50, replace=False)

# Plot negative images (as in your original)
for n in range(5):
    for m in range(10):
        idx = neg_selection[m + 10 * n]
        path = os.path.join(all_rays_dir, data.loc[idx, 'image_id'])
        image = mpimg.imread(path)
        ax[n, m].imshow(image)
        ax[n, m].grid(False)
        ax[n, m].axis('off')  # Hide axes ticks

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(5,10,figsize=(20,10))
for n in range(5):
    for m in range(10):
        idx = pos_selection[m + 10*n]
        path =os.path.join(all_rays_dir,data.loc[idx, 'image_id'])
        image = mpimg.imread(path)
        ax[n,m].imshow(image)
        ax[n,m].grid(False)

In [20]:
import os
import pandas as pd
import numpy as np

def extract_coords(df):
    coord = df.path.str.rsplit("_", n=4, expand=True)
    coord = coord.drop([0, 1, 4], axis=1)
    coord = coord.rename({2: "x", 3: "y"}, axis=1)
    coord["x"] = coord["x"].str.replace("x", "", case=False).astype(int)
    coord["y"] = coord["y"].str.replace("y", "", case=False).astype(int)
    df["x"] = coord.x.values
    df["y"] = coord.y.values
    return df

def get_cancer_dataframe(patient_id, cancer_id):
    path = os.path.join(cancer_rays_dir_str, patient_id, cancer_id)
    files = os.listdir(path)
    dataframe = pd.DataFrame(files, columns=["filename"])

    path_names = [os.path.join(path, f) for f in dataframe.filename.values]

    # Split coordinates
    coords = dataframe.filename.str.rsplit("_", n=4, expand=True)
    coords = coords[[2, 3]].rename({2: "x", 3: "y"}, axis=1)

    dataframe = dataframe.assign(
        x = coords["x"].str.replace("x", "", case=False).astype(int),
        y = coords["y"].str.replace("y", "", case=False).astype(int),
        target = int(cancer_id),
        path = path_names
    )

    return dataframe

def get_patient_dataframe(patient_id):
    df_0 = get_cancer_dataframe(patient_id, "0")
    df_1 = get_cancer_dataframe(patient_id, "1")
    patient_df = pd.concat([df_0, df_1], ignore_index=True)
    return patient_df

In [21]:
example = get_patient_dataframe(data.patient_id.values[0])
example.head()

,filename,x,y,target,path
0,10257_idx5_x1751_y551_class0.png,1751,551,0,/content/IDC_regular_ps50_idx5/10257/0/10257_i...
1,10257_idx5_x1401_y1301_class0.png,1401,1301,0,/content/IDC_regular_ps50_idx5/10257/0/10257_i...
2,10257_idx5_x2101_y1551_class0.png,2101,1551,0,/content/IDC_regular_ps50_idx5/10257/0/10257_i...
3,10257_idx5_x2251_y1551_class0.png,2251,1551,0,/content/IDC_regular_ps50_idx5/10257/0/10257_i...
4,10257_idx5_x1801_y551_class0.png,1801,551,0,/content/IDC_regular_ps50_idx5/10257/0/10257_i...


In [ ]:
# Ensure target is integer
data['target'] = data['target'].astype(int)

# Randomly select a negative patch (target=0)
random_image_idx = np.random.choice(data[data.target == 0].index, size=1, replace=False)
path = os.path.join(all_rays_dir, data.loc[random_image_idx[0], 'image_id'])

# Load and show image
image = mpimg.imread(path)
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
from skimage.filters import gaussian
import matplotlib.pyplot as plt
import numpy as np

# Convert image to float in [0,1]
if image.dtype == np.uint8:
    image_float = image.astype(np.float32) / 255.0
else:
    image_float = image.astype(np.float32)  # ensure float

# Apply Gaussian blur
gaussian_image = gaussian(image_float, sigma=1, channel_axis=-1)

# Clip to [0,1] to avoid any out-of-range values
gaussian_image = np.clip(gaussian_image, 0, 1)

# Display
plt.imshow(gaussian_image)
plt.axis('off')
plt.show()

In [ ]:
from skimage.util import random_noise
import matplotlib.pyplot as plt
import numpy as np

# Convert image to float in [0,1]
if image.dtype == np.uint8:
    image_float = image.astype(np.float32) / 255.0
else:
    image_float = image.astype(np.float32)

# Apply random noise
noise_image = random_noise(image_float, mode='gaussian', var=0.01)  # var=0.01 for small noise

# Clip to [0,1] just in case
noise_image = np.clip(noise_image, 0, 1)

# Display
plt.imshow(noise_image)
plt.axis('off')
plt.show()

In [ ]:
from skimage.util import random_noise
import matplotlib.pyplot as plt
import numpy as np

# Ensure gaussian_image is in [0,1]
gaussian_image_norm = np.clip(gaussian_image, 0, 1)

# Apply random Gaussian noise
noise_gaussian_image = random_noise(gaussian_image_norm, mode='gaussian', var=0.01)

# Clip again to [0,1] to be safe
noise_gaussian_image = np.clip(noise_gaussian_image, 0, 1)

# Display
plt.imshow(noise_gaussian_image)
plt.axis('off')
plt.show()

In [26]:
import os

# Create main folder
os.makedirs('image_processing', exist_ok=True)

# Create subfolder for noisy images
os.makedirs('image_processing/noise_images', exist_ok=True)

In [27]:
# Folder to save noisy images
save_folder = 'image_processing/noise_images'
os.makedirs(save_folder, exist_ok=True)

for normal_image in all_rays_dir_lst:
    path = os.path.join(all_rays_dir, normal_image)

    # Read image
    img = mpimg.imread(path)

    # Ensure float in [0,1]
    if img.dtype == np.uint8:
        img_float = img.astype(np.float32) / 255.0
    else:
        img_float = img.astype(np.float32)

    # Apply random noise
    noise_image = random_noise(img_float, mode='gaussian', var=0.01)

    # Clip to [0,1] just in case
    noise_image = np.clip(noise_image, 0, 1)

    # Save
    new_path = os.path.join(save_folder, normal_image)
    mpimg.imsave(new_path, noise_image)


In [28]:
import os

os.makedirs('image_processing/processd_data_train/zeros', exist_ok=True)
os.makedirs('image_processing/processd_data_train/ones', exist_ok=True)
os.makedirs('image_processing/processd_data_test/zeros', exist_ok=True)
os.makedirs('image_processing/processd_data_test/ones', exist_ok=True)

In [29]:
processd_lst = os.listdir('image_processing/noise_images')
processd_lst_str = 'image_processing/noise_images'
processd_data = pd.DataFrame(processd_lst, columns=['image_id'])
processd_data.head()

,image_id
0,10257_idx5_x1751_y551_class0.png
1,10268_idx5_x501_y601_class0.png
2,12906_idx5_x1851_y2401_class1.png
3,14156_idx5_x2351_y1651_class0.png
4,14079_idx5_x3101_y1201_class0.png


In [30]:
def extract_target(x):
    try:
        a = x.split('_')
        b = a[4]
        target = b[5]
        return int(target)   # convert directly to integer
    except IndexError:
        return -1  # fallback if filename format unexpected

processd_data['target'] = processd_data['image_id'].apply(extract_target)

processd_data.head(10)

,image_id,target
0,10257_idx5_x1751_y551_class0.png,0
1,10268_idx5_x501_y601_class0.png,0
2,12906_idx5_x1851_y2401_class1.png,1
3,14156_idx5_x2351_y1651_class0.png,0
4,14079_idx5_x3101_y1201_class0.png,0
5,10276_idx5_x1251_y1451_class1.png,1
6,10268_idx5_x1551_y451_class0.png,0
7,14155_idx5_x1701_y951_class1.png,1
8,16570_idx5_x1601_y601_class0.png,0
9,10272_idx5_x3351_y951_class0.png,0


In [31]:
processd_data['target'].value_counts()

,count
target,
0,198738
1,78786


In [32]:
from sklearn.model_selection import train_test_split

# Extract the target column (labels) from the dataframe
y = processd_data['target']

# Split the dataset into training and testing sets
# test_size=0.10 → 10% of the data will go to the test set, 90% to the train set
# random_state=101 → ensures the split is reproducible (same split every run)
# stratify=y → maintains the same class distribution of the target variable in both sets
processd_train, processd_test = train_test_split(
    processd_data,
    test_size=0.10,
    random_state=101,
    stratify=y
)

# Extract only the image_id column from the test dataframe
# These IDs will typically be used to locate or copy test images
processd_test_pls = processd_test.image_id

# Extract only the image_id column from the train dataframe
# These IDs will typically be used to locate or copy training images
processd_train_pls = processd_train.image_id

In [34]:
# Set 'image_id' as the index for processd_data to allow direct lookup by image filename
processd_data.set_index('image_id', inplace=True)

for image in processd_test_pls:
    fname = image

    # Access target using .loc with the index 'image_id'
    target = processd_data.loc[image, 'target']

    # Compare target with integers, as it's already an integer type
    if target == 0:
        label = 'zeros'
    elif target == 1:
        label = 'ones'
    # No 'else: continue' is needed if target is guaranteed to be 0 or 1

    src = os.path.join(processd_lst_str, fname)
    dst = os.path.join("image_processing/processd_data_test", label, fname)

    shutil.copyfile(src, dst)

In [35]:
for image in processd_train_pls:                 # Loop through all training image IDs
    fname  = image                               # Image file name
    target = processd_data.loc[image, 'target']  # Get the label (0 or 1) for this image

    if target == 0:
        label = 'zeros'                          # Folder name for class 0
    elif target == 1:
        label = 'ones'                           # Folder name for class 1

    src = os.path.join(processd_lst_str, fname)  # Source path of the image
    dst = os.path.join('image_processing/processd_data_train', label, fname)  # Destination path

    shutil.copyfile(src, dst)                    # Copy image to the class folder

In [36]:
print(len(os.listdir('image_processing/processd_data_train/zeros')))
print(len(os.listdir('image_processing/processd_data_train/ones')))
print(len(os.listdir('image_processing/processd_data_test/zeros')))
print(len(os.listdir('image_processing/processd_data_test/ones')))

178864
70907
19874
7879


In [37]:
processd_lst = os.listdir('image_processing/noise_images')
# Get list of all image file names from the folder

processd_lst_str = 'image_processing/noise_images'
# Store the folder path for later use

processd_data = pd.DataFrame(processd_lst, columns=['image_id'])
# Create a DataFrame with image file names

def extract_target(x):
    a = x.split('_')      # Split the filename using "_"
    b = a[4]              # Select the 5th part of the filename
    target = b[5]         # Extract the target value (0 or 1) from that part
    return target

processd_data['target'] = processd_data['image_id'].apply(extract_target)
# Apply the function to extract the target label for each image

processd_data.head(10)
# Display the first 10 rows of the DataFrame

,image_id,target
0,10257_idx5_x1751_y551_class0.png,0
1,10268_idx5_x501_y601_class0.png,0
2,12906_idx5_x1851_y2401_class1.png,1
3,14156_idx5_x2351_y1651_class0.png,0
4,14079_idx5_x3101_y1201_class0.png,0
5,10276_idx5_x1251_y1451_class1.png,1
6,10268_idx5_x1551_y451_class0.png,0
7,14155_idx5_x1701_y951_class1.png,1
8,16570_idx5_x1601_y601_class0.png,0
9,10272_idx5_x3351_y951_class0.png,0


In [38]:
os.mkdir( 'image_processing/model_tst')
os.mkdir( 'image_processing/model_tst/trainig')
os.mkdir( 'image_processing/model_tst/testing')
os.mkdir( 'image_processing/model_tst/trainig/zeros')
os.mkdir( 'image_processing/model_tst/trainig/ones')
os.mkdir( 'image_processing/model_tst/testing/zeros')
os.mkdir( 'image_processing/model_tst/testing/ones')

In [39]:
# Take 10,000 samples where target = '0'
df_0 = processd_data[processd_data['target'] == '0'].sample(10000, random_state=101)

# Take 10,000 samples where target = '1'
df_1 = processd_data[processd_data['target'] == '1'].sample(10000, random_state=101)

# Combine both classes into one dataset and reset the index
test_data = pd.concat([df_0, df_1], axis=0).reset_index(drop=True)

# Extract the target column for stratified splitting
test_y = test_data['target']

# Split the dataset into train (90%) and test (10%) while preserving class distribution
test_data_train, test_data_test = train_test_split(
    test_data,
    test_size=0.10,
    random_state=101,
    stratify=test_y
)

# Extract image IDs for training
sts_train = test_data_train.image_id

# Extract image IDs for testing
tst_test  = test_data_test.image_id

# Set image_id as the index for easier lookup later
test_data.set_index('image_id', inplace=True)

# ---------------- TRAIN LOOP ----------------
for image in sts_train:                     # Loop through training image IDs
    fname  = image                          # Image file name
    target = test_data.loc[image, 'target'] # Get label for the image

    if target == '0':
        label = 'zeros'                     # Folder name for class 0
    elif target == '1':
        label = 'ones'                      # Folder name for class 1

    src = os.path.join(all_rays_dir, fname) # Source image path
    dst = os.path.join('image_processing/model_tst/trainig', label, fname) # Destination path

    shutil.copyfile(src, dst)               # Copy image to training folder

# ---------------- TEST LOOP ----------------
for image in tst_test:                      # Loop through testing image IDs
    fname  = image
    target = test_data.loc[image, 'target'] # Get label for the image

    if target == '0':
        label = 'zeros'                     # Folder name for class 0
    elif target == '1':
        label = 'ones'                      # Folder name for class 1

    src = os.path.join(all_rays_dir, fname) # Source image path
    dst = os.path.join('image_processing/model_tst/testing', label, fname) # Destination path

    shutil.copyfile(src, dst)               # Copy image to testing folder

In [ ]:
# Convert the target column from string to integer (0 or 1)
processd_data['target'] = processd_data['target'].astype(int)

# Create a 5x4 grid of subplots to display 20 images
fig, ax = plt.subplots(5, 4, figsize=(30,20))

# Randomly select 20 indices where target = 1
pos_selection = np.random.choice(
    processd_data[processd_data.target == 1].index,
    size=20,
    replace=False
)

# Loop through the subplot grid
for n in range(5):
    for m in range(4):
        idx = pos_selection[m + 4*n]   # Get index for the current subplot

        fname = processd_data.loc[idx, 'image_id']  # Get image file name
        path = os.path.join(processd_lst_str, fname) # Create full image path

        image = mpimg.imread(path)     # Read the image
        ax[n, m].imshow(image)         # Display the image
        ax[n, m].grid(False)           # Turn off grid lines

In [ ]:
fig, ax = plt.subplots(5,4,figsize=(30,20))
for n in range(5):
    for m in range(4):
        idx = neg_selection[m + 4*n]
        path =os.path.join(processd_lst_str,processd_data.loc[idx, 'image_id'])
        image = mpimg.imread(path)
        ax[n,m].imshow(image)
        ax[n,m].grid(False)

In [42]:
data_processd_test_generation = ImageDataGenerator(rescale=1.0/255)
train_generation_processd = data_processd_test_generation.flow_from_directory("image_processing/model_tst/trainig", target_size=(50,50), batch_size=10,class_mode='categorical')
test_generation_processd = data_processd_test_generation.flow_from_directory("image_processing/model_tst/testing",target_size=(50,50),batch_size=10,class_mode='categorical')

Found 18000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


In [43]:
# Create a Sequential CNN model
my_model_im_processd = Sequential()

# First convolution layer:
# 32 filters of size 4x4, input image size 50x50 with 3 color channels (RGB)
# ReLU activation introduces non-linearity
my_model_im_processd.add(Conv2D(filters=32, kernel_size=(4,4), input_shape=(50,50,3), activation='relu'))

# Max pooling layer:
# Reduces spatial size of the feature maps (downsampling)
my_model_im_processd.add(MaxPool2D(pool_size=(2,2)))

# Flatten layer:
# Converts the 2D feature maps into a 1D vector for the dense layers
my_model_im_processd.add(Flatten())

# Fully connected dense layer with 128 neurons
# ReLU activation for learning complex patterns
my_model_im_processd.add(Dense(128, activation='relu'))

# Output layer with 2 neurons (for two classes)
# Softmax converts outputs into class probabilities
my_model_im_processd.add(Dense(2, activation='softmax'))

# Compile the model
# categorical_crossentropy → loss function for multi-class classification
# adam → optimizer used to update weights
# accuracy → evaluation metric
my_model_im_processd.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=2)

my_model_im_processd.fit(
    train_generation_processd,
    validation_data=test_generation_processd,
    epochs=60,
    verbose=1,
    callbacks=[early_stop]
)

In [45]:
os.mkdir( 'image_processing/normal')
os.mkdir( 'image_processing/normal/model_tst')
os.mkdir( 'image_processing/normal/model_tst/trainig')
os.mkdir( 'image_processing/normal/model_tst/testing')
os.mkdir( 'image_processing/normal/model_tst/trainig/zeros')
os.mkdir( 'image_processing/normal/model_tst/trainig/ones')
os.mkdir( 'image_processing/normal/model_tst/testing/zeros')
os.mkdir( 'image_processing/normal/model_tst/testing/ones')

In [46]:
# Create a DataFrame containing all image file names
data = pd.DataFrame(all_rays_dir_lst, columns=['image_id'])

# Extract the target label from each file name
data['target'] = data['image_id'].apply(extract_target)

# Randomly sample 10,000 images where target = '0'
df_0 = data[data['target'] == '0'].sample(10000, random_state=101)

# Randomly sample 10,000 images where target = '1'
df_1 = data[data['target'] == '1'].sample(10000, random_state=101)

# Combine both classes into one dataset and reset the index
test_data = pd.concat([df_0, df_1], axis=0).reset_index(drop=True)

# Extract target column for stratified splitting
test_y = test_data['target']

# Split the dataset into training (90%) and testing (10%) while preserving class balance
test_data_train, test_data_test = train_test_split(
    test_data,
    test_size=0.10,
    random_state=101,
    stratify=test_y
)

# Extract image IDs for training set
sts_train = test_data_train.image_id

# Extract image IDs for testing set
tst_test  = test_data_test.image_id

# Set image_id as index for easier label lookup
test_data.set_index('image_id', inplace=True)

# ---------------- TRAIN LOOP ----------------
for image in sts_train:                         # Loop through training image IDs
    fname  = image                              # Image file name
    target = test_data.loc[image, 'target']     # Get label for the image

    if target == '0':
        label = 'zeros'                         # Folder name for class 0
    elif target == '1':
        label = 'ones'                          # Folder name for class 1

    src = os.path.join(all_rays_dir, fname)     # Source image path
    dst = os.path.join('image_processing/normal/model_tst/trainig', label, fname)  # Destination path

    shutil.copyfile(src, dst)                   # Copy image to training folder

# ---------------- TEST LOOP ----------------
for image in tst_test:                          # Loop through testing image IDs
    fname  = image
    target = test_data.loc[image, 'target']     # Get label for the image

    if target == '0':
        label = 'zeros'                         # Folder name for class 0
    elif target == '1':
        label = 'ones'                          # Folder name for class 1

    src = os.path.join(all_rays_dir, fname)     # Source image path
    dst = os.path.join('image_processing/normal/model_tst/testing', label, fname)  # Destination path

    shutil.copyfile(src, dst)                   # Copy image to testing folder

In [47]:
data_normal_test_generation = ImageDataGenerator(rescale=1.0/255)
train_generation_normal = data_normal_test_generation.flow_from_directory("image_processing/normal/model_tst/trainig", target_size=(50,50), batch_size=10,class_mode='categorical')
test_generation_normal = data_normal_test_generation.flow_from_directory("image_processing/normal/model_tst/testing",target_size=(50,50),batch_size=10,class_mode='categorical')

Found 18000 images belonging to 2 classes.
Found 2000 images belonging to 2 classes.


In [48]:
# Create a Sequential CNN model
my_model_im_norm = Sequential()

# First convolution layer:
# 32 filters of size 4x4 applied to 50x50 RGB images
# ReLU activation introduces non-linearity
my_model_im_norm.add(Conv2D(filters=32, kernel_size=(4,4), input_shape=(50,50,3), activation='relu'))

# Max pooling layer to reduce spatial dimensions
my_model_im_norm.add(MaxPool2D(pool_size=(2,2)))

# Flatten the 2D feature maps into a 1D vector
my_model_im_norm.add(Flatten())

# Fully connected layer with 128 neurons
my_model_im_norm.add(Dense(128, activation='relu'))

# Output layer with 2 neurons for binary classification
# Softmax converts outputs to class probabilities
my_model_im_norm.add(Dense(2, activation='softmax'))

# Compile the model
# categorical_crossentropy → loss function for one-hot encoded labels
# adam → optimizer for weight updates
# accuracy → evaluation metric
my_model_im_norm.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [49]:
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

my_model_im_norm.fit(
    train_generation_normal,
    validation_data=test_generation_normal,
    epochs=60,
    verbose=1,
    callbacks=[early_stop]
)

Epoch 1/60


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1800/1800 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.6915 - loss: 0.6240 - val_accuracy: 0.7460 - val_loss: 0.5225
Epoch 2/60
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.7615 - loss: 0.5063 - val_accuracy: 0.7830 - val_loss: 0.4736
Epoch 3/60
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.7880 - loss: 0.4724 - val_accuracy: 0.7665 - val_loss: 0.4854
Epoch 4/60
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 20s 7ms/step - accuracy: 0.7844 - loss: 0.4709 - val_accuracy: 0.7805 - val_loss: 0.4694
Epoch 5/60
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.8021 - loss: 0.4458 - val_accuracy: 0.7955 - val_loss: 0.4554
Epoch 6/60
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.8012 - loss: 0.4435 - val_accuracy: 0.7670 - val_loss: 0.4947
Epoch 7/60
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 19s 11ms/step - accuracy: 0.8068 - loss: 0.4327 - val_accuracy: 0.7860 - val_loss: 0.4590


In [ ]:
losse = pd.DataFrame(my_model_im_norm.history.history)
losse.head()

In [ ]:
losse[['accuracy','val_accuracy']].plot()

In [ ]:
losse[['loss','val_loss']].plot()